# 02. Обучение LoRA и повторное извлечение векторов
Для обучения требуется GPU. Имя запуска, признак и каталог результатов должны совпадать с настройками ноутбука 01.

In [ ]:
from hashlib import sha256
from pathlib import Path
from runpy import run_path
from subprocess import run

REPO_URL = "https://github.com/danilov-e/lora-activation-steering.git"
RUNS_DIR = Path("/content/drive/MyDrive/lora_activation_steering/runs")
RUN_NAME = "research_01"
TRAIT = "sycophancy"

repo_id = sha256(REPO_URL.encode()).hexdigest()[:12]
source = Path("/content/lora-steering") / repo_id / "source"
if not source.exists():
    source.parent.mkdir(parents=True, exist_ok=True)
    run(["git", "clone", "--quiet", "--depth", "1", REPO_URL, str(source)], check=True)
else:
    run(["git", "-C", str(source), "fetch", "--quiet", "origin", "HEAD"], check=True)
    run(["git", "-C", str(source), "checkout", "--quiet", "--detach", "FETCH_HEAD"], check=True)

colab = run_path(str(source / "notebooks" / "colab.py"))
PROJECT, RUNS, _ = colab["setup"](source, RUNS_DIR, RUN_NAME, TRAIT, resume=True)

from src import report as r, workflow as w
from IPython.display import Image, display
import pandas as pd

In [ ]:
RUN = w.open_run(PROJECT, RUN_NAME, TRAIT, runs_dir=RUNS)
choices = w.load_choices(RUN)
print(RUN)
display(pd.DataFrame(choices))

## 1. Обучение адаптеров
Все адаптеры обучаются на ответах `neg`. Одна LoRA обучается без стиринга, остальные — с фиксированным исходным вектором на выбранном слое. После обучения векторы извлекаются при выключенном стиринге.

In [ ]:
labels = w.train_all(RUN)
display(Image(filename=str(r.validation_loss(RUN, labels))))

## 2. Сравнение векторов
Для каждого адаптера рассчитываются отношение норм и косинусное сходство нового и исходного векторов.

In [ ]:
for label in labels:
    geometry, plot = r.compare_vectors(RUN, label)
    print(label)
    display(geometry.loc[geometry.layer.isin([c["layer"] for c in choices])].round(4))
    display(Image(filename=str(plot)))